# Benchmark: Split Flow

In [26]:
import rioxarray as rxr
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0,"/home/jovyan/shared/Libraries")
import victor
import cartopy.crs as ccrs
import subprocess
import os
import pandas as pd
import xarray as xr
import rasterio as rio
import glob

#### This benchmark is based on an experiment performed at the Syracuse University Lava Project that involved pouring molten basalt at a constant supply rate onto a sloping plane (Dietterich et al., [2015](https://appliedvolc.biomedcentral.com/articles/10.1186/s13617-017-0061-x#ref-CR20)).

This simulation captures both thermal effects and their impacts on flow rheology. This experiment also allows us to compare the flow propagation and surface temperature between the numerical simulations and the experimental data.

We encourage changing the volume betweens runs to compare the physical or stochastic reactions at various scales. It is also recommended to change the slope of the bed to measure various internal friction effects. Finally, the angle of the obstacle, which can dictate if/how the separated flows recombine.

If `custom_run` is set to `False`, a DEM that has been preformatted will be used, and the paramters will be ignored. If `True`, a new DEM will be dynamically generated using the provided parameters.

In [6]:
custom_run = False

bed_slope = 14

volume = 7.7e-4

split_angle = 90

#### The following cells should not be edited, unless you have extensive knowledge of the model.

The following cell sets up common parameters. Following this, the next 4 cells format and generate input files for each respective model, at parameters considered reasonable for this benchmark.

In [13]:
easting = 0
northing = 0
scale = volume/7.7e-4
dem = "DEMs_Lava_Benchmark/topography_dem_split.asc"
if custom_run:
    f = open("DEMs_Lava_Benchmark/slope.asc","r+")
    header = f.readlines()
    header[3] = f"""xllcorner {-2*scale}\n"""
    header[4] = f"""yllcorner {-1.5*scale}\n"""
    header[5] = f"""cellsize {0.01*scale}\n"""
    f.seek(0)
    f.writelines(header)
    f.truncate()
    f.close()
    
    r = rxr.open_rasterio("DEMs_Lava_Benchmark/slope.asc")
    height = r.rio.resolution()[0]*501*np.sin(np.deg2rad(bed_slope))
    slope = np.linspace(height,0,501)
    stacked = np.tile(slope,(301,1))
    r[0,:,:] = stacked
    y = 150
    x = 70
    for i in range(14):
        j = int(np.round(np.sin(np.deg2rad(split_angle))))
        r[0,y+j,x+j] = 5000
        r[0,y-j,x+j] = 5000
    r.rio.to_raster('Benchmark_Split.asc')
else:
    r = rxr.open_rasterio(dem)
xll, yll = int(r.x.min()), int(r.y.min())
coordinates = np.array([int(easting),int(northing)])
res = r.rio.resolution()[0]

RasterioIOError: DEMs_Lava_Benchmark/topography_dem_split.asc: No such file or directory

### Molasses: Additional Notes

Molasses input parameters are stored in `custom_molasses.conf` in the `Molasses` folder. The most impactful change would be that of changing the minimum and maximum residual, as the cellular automata structure will cause a much larger or smaller area to be impacted. As a stochastic model, no physical parameters are used.

In [19]:
os.chdir('/home/jovyan/Lava Benchmark/Molasses')

f = open("custom_molasses.conf","r+")
conf = f.readlines()
conf[2] = f"""MIN_RESIDUAL = {7e-6*scale}\n"""
conf[3] = f"""MAX_RESIDUAL ={1e-3*scale}\n"""
conf[4] = f"""MIN_TOTAL_VOLUME = {str(volume)}\n"""
conf[5] = f"""MAX_TOTAL_VOLUME = {str(volume)}\n"""
conf[6] = f"""MIN_PULSE_VOLUME = {float(volume)/100}\n"""
conf[7] = f"""MAX_PULSE_VOLUME = {float(volume)/100}\n"""
conf[10] = f"""DEM_FILE = ../{dem}\n"""
f.seek(0)
f.writelines(conf)
f.truncate()
f.close()

f=open("events.in","w")
f.write(f"""{easting},{northing}""")
f.close()

subprocess.run("molasses custom_molasses.conf",shell=True, stderr=subprocess.DEVNULL)

victor.convert_molasses("molasses_split",resolution=res)

Seeding random number generator: 1775162678


               MOLASSES is a lava flow simulator.

Config file: custom_molasses.conf
Reading in Parameters...
             PARENTS = 1                    [assigned]
    ELEVATION_UNCERT = 0                    [assigned]
        MIN_RESIDUAL = 7e-06                [assigned]
        MAX_RESIDUAL = 0.001                [assigned]
    MIN_TOTAL_VOLUME = 0.00077              [assigned]
    MAX_TOTAL_VOLUME = 0.00077              [assigned]
    MIN_PULSE_VOLUME = 7.699999999999999e-06 [assigned]
    MAX_PULSE_VOLUME = 7.699999999999999e-06 [assigned]
                RUNS = 1                    [assigned]
      ASCII_FLOW_MAP = 1                    [assigned]
            DEM_FILE = ../DEMs_Lava_Benchmark/topography_dem_split.asc [assigned]
         EVENTS_FILE = events.in            [assigned]
Nothing missing.

DEM Information [Arc/Info ASCII Grid]:
  File:              ../DEMs_Lava_Benchmark/topography_dem_split.asc
  Lower Left Origin: (-0.3515

### MrLavaLoba/Flowy: Additional Notes

MrLavaLoba/Flowy (a drop in C++ replacement) input parameters are stored in `input.toml` in the `flowy` folder. The most impactful changes would be those such as the thickening parameter, lobe exponent, or max slope probability. Other parameters not mentioned will also affect the creation and placement of lobes. As a stochastic model, no physical parameters are specified by the user.

In [22]:
os.chdir('/home/jovyan/Lava Benchmark/mr_lava_loba/')
g=open("input_data.py","r+")
inp = g.readlines()
inp[1] = """run_name = 'Split'\n"""
inp[3] = f"""source = '../{dem}'\n"""
inp[8] = "east_to_vent = 3.0\n"
inp[9] = "west_to_vent = 3.0\n"
inp[10] = "south_to_vent = 3.0\n"
inp[11] = "north_to_vent = 3.0\n"
inp[41] = """vent_flag = 0\n"""
inp[43] = f"""x_vent = [0]\n"""
inp[44] = f"""y_vent = [0]\n"""
inp[74] = f"""n_flows = 30\n"""
inp[79] = f"""min_n_lobes = 5000\n"""
inp[80] = f"""max_n_lobes = 5000\n"""
inp[98] = f"""lobe_area = .0002\n"""
inp[108] = f"""thickness_ratio = 1.25\n"""
inp[118] = f"""thickening_parameter = 0.75\n"""
inp[130] = f"""lobe_exponent = .03\n"""
inp[140] = f"""max_slope_prob = 0.92\n"""
inp[148] = f"""inertial_exponent = 0.4\n"""
inp[90] = f"""total_volume = {volume}\n"""
g.seek(0)
g.writelines(inp)
g.truncate()
g.close()

subprocess.run("python mr_lava_loba.py",shell=True)


Mr Lava Loba by M.de' Michieli Vitturi and S.Tarquini

x_vent_end not used
y_vent_end not used
fissure_probabilities not used
Run name Split_001

Maximum number of lobes 5000
Average Lobe thickness = 0.000036 m

west_to_vent 5.0
x_vent [0]
Crop flag =  True
../DEMs_Lava_Benchmark/topography_dem_split.npy  does not exist
Cropping of original DEM
xW,xE,yS,yN -5.0 5.0 -5.0 5.0
iW,iE,jS,jN 0 501 0 301

Time to read DEM 0.13185946699999995s
Channel parameters not defined:
- channel_file
- d1
- d2
- eps
- alfa_chaneel

Restart_files not used
max_semiaxis 0.012615662610100801
max_cells 11
End pre-processing

[====================] 100%0:00:01

Total number of lobes 150000 Average number of lobes 5000

Time elapsed 2.7818944839999995 sec. / 0 min.

Saving files

Split_001_thickness_full.asc saved
Union_diff_file not defined

Masking threshold 0.96
Total volume 3.7331655193711e-05  m3 Masked volume 3.573061147190292e-05  m3 Volume fraction 0.9571129725296029
Total area 0.252855  m2 Masked area

CompletedProcess(args='python mr_lava_loba.py', returncode=0)

In [25]:
os.chdir('/home/jovyan/Lava Benchmark/IMEX_LavaFlow')
radius = 2e-3/scale
subprocess.run("mv IMEX_LavaFlow_bm4.inp IMEX_LavaFlow.inp", shell=True)
h=open("IMEX_LavaFlow.inp","r+")
inp = h.readlines()
inp[100] = f""" VEL_SOURCE= {.4713*scale}\n"""
h.seek(0)
h.writelines(inp)
h.truncate()
h.close()

p = subprocess.Popen(['./IMEX_LavaFlow'], stdin=subprocess.PIPE, shell=True)
p.communicate(input=b'\n')
subprocess.run("mv IMEX_LavaFlow.inp IMEX_LavaFlow_bm4.inp", shell=True)

mv: cannot stat 'IMEX_LavaFlow_bm4.inp': No such file or directory


 ---------------------
 IMEX_LavaFlow 1.0
 ---------------------
 OpenMP program
 Number of threads used           4
 Run name: BM4_split                               
 ----- 2D SIMULATION -----
 Ambient density =    1.1763298740177413       (kg/m3)
 CARRIER PHASE: liquid
 Carrier phase kinematic viscosity:  -1.0000000000000000     
 VISC_PAR =   0.0000000000000000     
 Press ENTER to continue
 radiative_term_coeff =    2.2680000000000002E-008
 convective_term_coeff =    35.000000000000000     
 Searching for DEM file
 Reading DEM file
 ncols         501
 nrows         301
 Percent Complete: 100.00% 
 IOSTAT=          -1
 thickness_levels          -1
 dyn_pres_levels          -1
 Searching for probes coords
 Source area =   3.1399998813867573E-006  Error =   5.0699513866514501E-004
 Implicit equations =           3

 ******** START COMPUTATION *********

 WRITING BM4_split_0000.p_2d                     
 WRITING BM4_split_0000.asc                      
 WRITING BM4_split_T_0000.asc  

KeyboardInterrupt: 

In [ ]:
scale = ratio
os.chdir('/home/jovyan/lava2d_edited/')

subprocess.run("python example_BM4_split.py", shell=True)
subprocess.run("python convert_output.py outputs/out.nc -o lava2d_split.asc", shell=True)

#### The final two cells concern visualization.

The first provides a reference to the final timestamp of each simulation, though all outputs are avilable in their models' respective folder. The second and last cell plots each model side by side. We encourage users to edit this to their personal preference.

In [ ]:
os.chdir("/home/jovyan/Lava Benchmark/")

mll_outputs = glob.glob(f'mr_lava_loba/Split*thickness_masked*.asc')
mll_out = mll_outputs[-1]

r = rxr.open_rasterio("IMEX_LavaFlow/Split_max.asc",masked=True)
r2 = rxr.open_rasterio("Molasses/molasses_split.asc",masked=True)
r3 = rxr.open_rasterio(mll_out,masked=True)
r4 = rxr.open_rasterio("lava2d/lava2d_split.as",masked=True)
r_min, r_max = r.min(), r.max()
r2_min, r2_max = r2.min(), r2.max()
r3_min, r3_max = r3.min(), r3.max()
r4_min, r4_max = r4.min()/1000, r4.max()/1000

maximum = np.max((r_max,r2_max,r3_max,r4_max))
idx_max = np.argmax((r_max,r2_max,r3_max,r4_max))

In [ ]:
fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(ncols=2, nrows=2,subplot_kw=dict(projection=ccrs.epsg(32628)), figsize = (12,9))

xticks = np.linspace(-.2, 4.81, 5)
yticks = np.linspace(-1.5, 1.51, 5)

fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(ncols=2, nrows=2, figsize = (12,9))

thickness0 = victor.plot_flow(dem, "Molasses/molasses_split.asc", axes=ax0,zoom=False,title='Molasses',minimum=1e-6,scale=maximum)
thickness1 = victor.plot_flow(dem, flowy_out, axes=ax1, zoom=False,title='MrLavaLoba/Flowy',minimum=1e-6, scale=maximum)
thickness2 = victor.plot_flow(dem, "IMEX_LavaFlow/Split_max.asc", axes=ax2, zoom=False,title='IMEX_LavaFlow', minimum=1e-6, scale=maximum)
thickness3 = victor.plot_flow(dem, "lava2d_split.asc", axes=ax3 ,title='Lava2D', minimum=1e-10, scale=maximum*100)

cbar_ax = fig.add_axes([.25, 0, 0.5, 0.05])
thickness = [thickness0, thickness1, thickness2, thickness3]
thickness = thickness[idx_max]
fig.colorbar(thickness, cax=cbar_ax, label="Flow Thickness (m)", orientation="horizontal")